# Ransomware leak-site market analysis

Replication notebook for the empirical analysis. It uses the claims published on ransomware data leak sites (DLS) and aggregated by [ransomware.live](https://www.ransomware.live) through its public API v2 (`/victims/<year>/<month>`, no key required).

**Questions**
1. Is the RaaS market concentrated? How do HHI and CR5 evolve over time?
2. How long do brands survive, and how many enter and exit each quarter?
3. What happens after a takedown (LockBit, Operation Cronos): do claims fall, or are they redistributed to other brands?
4. Are groups specialised by sector and country? What is the weight of the EU and Italy?

**Limitations**: leak sites only list victims that did not pay (or were not yet paid); claims can be false, duplicated or reposted; `discovered` is the date the post was found, not the date of the attack. Sector and country are assigned by the aggregator and are incomplete.

Structure: 0 Setup · 1 Cached download · 2 Cleaning · 3 Overview · 4 Concentration · 5 Brand dynamics · 6 Takedowns · 7 Sectors and countries · 8 Export · 9 Robustness

## 0. Setup

In [ ]:
from pathlib import Path
import json, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import statsmodels.formula.api as smf

DATA_DIR = Path("data"); RAW_DIR = DATA_DIR / "raw"
FIG_DIR = Path("figures"); TAB_DIR = Path("tables")
for d in (RAW_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

API_BASE = "https://api.ransomware.live/v2"
START = "2022-01"                                                       # first month included
END = (pd.Timestamp.today().to_period("M") - 1).strftime("%Y-%m")       # last complete month
SLEEP_S = 2.0            # pause between requests (free API)
INACTIVE_DAYS = 90       # a brand with no post for >90 days has exited
MIN_VICTIMS_BRAND = 3    # minimum claims for the survival analysis

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "figure.figsize": (9, 4.2),
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.3, "font.size": 10, "legend.frameon": False,
})
PALETTE = ["#2a6fdb", "#e0703a", "#2a9d8f", "#c7408b", "#8a6bd1",
           "#d4a017", "#5c8a3a", "#b5453d", "#4aa3c7", "#7a7a7a"]

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight")

print(f"Period: {START} → {END}")

## 1. Cached download

Each month is saved to `data/raw/YYYY-MM.json`, so a second run downloads nothing. To refresh a month (posts can arrive late), use `force=True`. The aggregator is corrected over time, so the download date matters for replication (ours: 24 October 2026).

In [ ]:
def fetch_month(period: str, force: bool = False) -> list:
    path = RAW_DIR / f"{period}.json"
    if path.exists() and not force:
        return json.loads(path.read_text())
    y, m = period.split("-")
    url = f"{API_BASE}/victims/{y}/{m}"
    for attempt in range(4):
        r = requests.get(url, headers={"accept": "application/json"}, timeout=90)
        if r.status_code == 429:                     # rate limit: increasing back-off
            time.sleep(30 * (attempt + 1)); continue
        r.raise_for_status()
        data = r.json()
        if isinstance(data, dict):                   # some versions wrap the records
            data = data.get("victims") or data.get("data") or []
        path.write_text(json.dumps(data))
        time.sleep(SLEEP_S)
        return data
    raise RuntimeError(f"Persistent rate limit on {period}")

periods = pd.period_range(START, END, freq="M").strftime("%Y-%m")
raw = []
for p in periods:
    recs = fetch_month(p)
    for rec in recs:
        rec["_query_month"] = p
    raw.extend(recs)

(DATA_DIR / "download_log.json").write_text(json.dumps(
    {"downloaded_at": pd.Timestamp.now().isoformat(), "start": START, "end": END,
     "records": len(raw)}, indent=2))
print(f"{len(raw):,} raw records in {len(periods)} months")
print("Fields of the first record:", sorted(raw[0].keys()))

## 2. Cleaning and normalisation

Field names changed between API versions: `FIELD_MAP` accepts both. Check the output of the previous cell and extend the map if new names appear.

`GROUP_ALIASES` merges spelling variants of the same brand (e.g. `lockbit3`, `lockbit5` → `lockbit`). Documented rebrands are handled separately in `robustness_step2.py`.

In [ ]:
FIELD_MAP = {
    "victim":     ["victim", "post_title"],
    "group":      ["group", "group_name"],
    "discovered": ["discovered"],
    "attackdate": ["attackdate", "published"],
    "country":    ["country"],
    "sector":     ["activity", "sector"],
    "domain":     ["domain", "website"],
}
MISSING = {"not found", "unknown", "n/a", "na", "none", "", "null"}

def pick(rec, keys):
    for k in keys:
        v = rec.get(k)
        if v is not None and str(v).strip().lower() not in MISSING:
            return v
    return None

df = pd.DataFrame([{**{k: pick(r, ks) for k, ks in FIELD_MAP.items()},
                    "query_month": r["_query_month"]} for r in raw])

GROUP_ALIASES = {            # regex → canonical name
    r"^lockbit.*":          "lockbit",
    r"^(alphv|blackcat).*": "alphv",
    r"^ransomhub.*":        "ransomhub",
    r"^everest.*":          "everest",
}
def norm_group(g):
    g = str(g).strip().lower()
    for pat, canon in GROUP_ALIASES.items():
        if re.match(pat, g):
            return canon
    return g

def to_dt(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_localize(None)

df["group"] = df["group"].map(norm_group, na_action="ignore")
df["country"] = df["country"].astype("string").str.strip().str.upper()
df["sector"] = df["sector"].astype("string").str.strip()
df["discovered"] = to_dt(df["discovered"])
df["attackdate"] = to_dt(df["attackdate"])
# Victim key: domain if valid and specific, otherwise normalised name.
# Generic domains (gov.br, co.uk...) and placeholder posts must not collide across victims.
dom = (df["domain"].astype("string").str.lower().str.strip()
       .str.replace(r"^https?://(www\.)?", "", regex=True).str.split("/").str[0])
valid_dom = (dom.str.match(r"^[a-z0-9-]+(\.[a-z0-9-]+)+$", na=False)
             & ~dom.str.match(r"^(gov|com|co|org|net|edu|ac|go|or)\.[a-z]{2}$", na=False)
             & (dom.str.len() >= 5))
name_key = df["victim"].astype(str).str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
df["victim_key"] = np.where(valid_dom, dom, name_key)
placeholder = (df["victim"].astype(str).str.contains(
    r"removed following a request|redacted|confidential|^\W*$", case=False, regex=True)
    | (df["victim_key"].str.len() < 4))
df.loc[placeholder, "victim_key"] = "_anon_" + df.index[placeholder].astype(str)

n_raw = len(df)
df = df.dropna(subset=["discovered", "group"])
n_nodate = n_raw - len(df)
# same group + same victim = repost of the same claim
df = df.sort_values("discovered").drop_duplicates(subset=["group", "victim_key"], keep="first")
n_dup = n_raw - n_nodate - len(df)
# same victim claimed by different groups (re-extortion, rebrands)
multi_group = df.groupby("victim_key")["group"].nunique()
df["claimed_by_multiple_groups"] = df["victim_key"].map(multi_group) > 1
# first claim of the victim by any group (robustness to groups reposting others' leaks)
df["first_claim"] = ~df.duplicated(subset=["victim_key"], keep="first")

df["month"] = df["discovered"].dt.to_period("M")
df["quarter"] = df["discovered"].dt.to_period("Q")
df = df[(df["month"] >= pd.Period(START, "M")) & (df["month"] <= pd.Period(END, "M"))]

print(f"Raw records: {n_raw:,} | without date/group: {n_nodate:,} | duplicates removed: {n_dup:,} | final: {len(df):,}")
print(f"Victims claimed by more than one group: {df['claimed_by_multiple_groups'].mean():.1%}")
reposters = (df[~df["first_claim"]]["group"].value_counts().head(8).rename("reposted_claims")
             .to_frame().join(df["group"].value_counts().rename("all_claims")))
reposters["repost_share"] = reposters["reposted_claims"] / reposters["all_claims"]
print("Groups with the most claims already made by others:\n", reposters.round(2).to_string())
missing = df[["country", "sector", "attackdate", "domain"]].isna().mean().rename("share_missing")
missing.to_frame().style.format("{:.1%}")

## 3. Panoramica

In [ ]:
EVENTS = {  # dates from official sources (DOJ, Europol, NCA)
    "Hive takedown":        "2023-01-26",
    "ALPHV disruption":     "2023-12-19",
    "LockBit / Op. Cronos": "2024-02-20",
    "ALPHV exit scam":      "2024-03-05",
}
EVENTS = {k: pd.Timestamp(v) for k, v in EVENTS.items()}

def add_events(ax, label=True):
    lo, hi = ax.get_ylim()
    for i, (name, d) in enumerate(EVENTS.items()):
        ax.axvline(d, color="#999", lw=0.8, ls="--", zorder=0)
        if label:   # staggered labels to avoid overlap
            y = hi - (hi - lo) * (0.04 + 0.09 * (i % 4))
            ax.text(d, y, " " + name, va="top", ha="left", fontsize=7, color="#555",
                    bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))

monthly = df.groupby("month").size().rename("victims")
active = df.groupby("month")["group"].nunique().rename("active_groups")
overview = pd.concat([monthly, active], axis=1)
overview.index = overview.index.to_timestamp()

fig, ax1 = plt.subplots()
ax1.plot(overview.index, overview["victims"], color=PALETTE[0], lw=1.8, label="Victims posted")
ax1.set_ylabel("Victims per month")
ax2 = ax1.twinx()
ax2.plot(overview.index, overview["active_groups"], color=PALETTE[1], lw=1.2, label="Active groups")
ax2.set_ylabel("Active groups"); ax2.grid(False); ax2.spines["right"].set_visible(True)
add_events(ax1)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
ax1.set_title("Leak-site victims and active ransomware groups per month")
savefig(fig, "fig1_monthly_overview"); plt.show()

yearly = df.groupby(df["discovered"].dt.year).agg(victims=("victim_key", "size"),
                                                  groups=("group", "nunique"))
yearly

## 4. Market concentration

For each quarter: share of claims by group, **HHI** (sum of squared shares × 10,000), **CR1/CR5** and the equivalent number of equal-sized firms (10,000/HHI). As a reference, the 2023 U.S. Merger Guidelines treat markets with HHI above 1,800 as highly concentrated. Note that "market share" here is the share of published claims, not of revenue.

In [ ]:
q = df.groupby(["quarter", "group"]).size().rename("n").reset_index()
q["share"] = q["n"] / q.groupby("quarter")["n"].transform("sum")
conc = q.groupby("quarter").agg(
    victims=("n", "sum"),
    active_groups=("group", "nunique"),
    hhi=("share", lambda s: (s ** 2).sum() * 10_000),
    cr1=("share", "max"),
    cr5=("share", lambda s: s.nlargest(5).sum()),
)
conc["equiv_firms"] = 10_000 / conc["hhi"]
conc["leader"] = q.loc[q.groupby("quarter")["n"].idxmax()].set_index("quarter")["group"]
conc.index = conc.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(conc.index, conc["hhi"], marker="o", ms=3, color=PALETTE[0])
axes[0].axhline(1800, color="#b5453d", lw=0.8, ls=":")
axes[0].text(conc.index[0], 1800, " HHI 1,800", va="bottom", fontsize=8, color="#b5453d")
axes[0].set_ylabel("HHI (0–10,000)")
axes[0].set_title("Market concentration of leak-site claims by quarter")
axes[1].plot(conc.index, conc["cr5"] * 100, marker="o", ms=3, color=PALETTE[2], label="CR5")
axes[1].plot(conc.index, conc["cr1"] * 100, marker="o", ms=3, color=PALETTE[1], label="CR1 (leader)")
axes[1].set_ylabel("Share of victims (%)"); axes[1].legend()
for ax in axes:
    add_events(ax, label=(ax is axes[0]))
savefig(fig, "fig2_concentration"); plt.show()

# Robustness: same measure counting only each victim's first claim
q1 = df[df["first_claim"]].groupby(["quarter", "group"]).size().rename("n").reset_index()
q1["share"] = q1["n"] / q1.groupby("quarter")["n"].transform("sum")
conc["hhi_first_claims"] = q1.groupby("quarter")["share"].apply(lambda s: (s ** 2).sum() * 10_000).values
conc.round({"hhi": 0, "hhi_first_claims": 0, "cr1": 3, "cr5": 3, "equiv_firms": 1})

In [ ]:
# Composition: top-8 groups by total volume + "Other" (quarterly shares sum to 100%)
TOP_K = 8
top_groups = df["group"].value_counts().head(TOP_K).index.tolist()
comp = (q.assign(g=np.where(q["group"].isin(top_groups), q["group"], "other"))
          .pivot_table(index="quarter", columns="g", values="share", aggfunc="sum").fillna(0))
comp = comp[top_groups + ["other"]]
comp.index = comp.index.to_timestamp()

fig, ax = plt.subplots(figsize=(9, 4.6))
colors = PALETTE[:TOP_K] + ["#d9d9d9"]
ax.stackplot(comp.index, (comp * 100).T.values, labels=comp.columns, colors=colors, alpha=0.9)
ax.set_ylim(0, 100); ax.set_ylabel("Share of quarterly victims (%)")
ax.set_title(f"Market shares of the top {TOP_K} groups")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
add_events(ax, label=False)
savefig(fig, "fig3_market_shares"); plt.show()

## 5. Brand dynamics: entry, exit, survival

A brand *enters* at its first post and *exits* if it has not posted for more than `INACTIVE_DAYS` days at the end of the observation period. Brands already active in the first month are **left-truncated** (they were born earlier) and are excluded from the survival estimate; brands still active are **right-censored**.

In [ ]:
obs_end = df["discovered"].max()
life = df.groupby("group").agg(first=("discovered", "min"), last=("discovered", "max"),
                               victims=("victim_key", "size"))
life = life[life["victims"] >= MIN_VICTIMS_BRAND].copy()
life["duration_days"] = (life["last"] - life["first"]).dt.days.clip(lower=1)
life["exited"] = (obs_end - life["last"]).dt.days > INACTIVE_DAYS
life["left_truncated"] = life["first"] < pd.Timestamp(START) + pd.DateOffset(months=1)

entries = life.groupby(life["first"].dt.to_period("Q")).size().rename("entries")
exits = life[life["exited"]].groupby(life.loc[life["exited"], "last"].dt.to_period("Q")).size().rename("exits")
flow = pd.concat([entries, exits], axis=1).fillna(0).astype(int)
flow = flow[flow.index > pd.Period(START, "Q")]          # the first quarter contains only spurious entries
flow.index = flow.index.to_timestamp()

fig, ax = plt.subplots()
w = 35
ax.bar(flow.index - pd.to_timedelta(w / 2, unit="D"), flow["entries"], width=w, color=PALETTE[2], label="Entries")
ax.bar(flow.index + pd.to_timedelta(w / 2, unit="D"), -flow["exits"], width=w, color=PALETTE[7], label="Exits")
ax.axhline(0, color="#333", lw=0.8)
ax.set_ylabel("Brands per quarter (exits negative)"); ax.legend()
ax.set_title(f"Brand entries and exits (≥{MIN_VICTIMS_BRAND} victims; exit = >{INACTIVE_DAYS} days silent)")
savefig(fig, "fig4_brand_flows"); plt.show()

print(f"Brands: {len(life)} | exited: {life['exited'].sum()} | left-truncated: {life['left_truncated'].sum()}")

In [ ]:
# Kaplan-Meier for brands born inside the observation window
try:
    from lifelines import KaplanMeierFitter
except ImportError:
    raise ImportError("pip install lifelines")

surv = life[~life["left_truncated"]]
kmf = KaplanMeierFitter()
kmf.fit(surv["duration_days"] / 30.44, event_observed=surv["exited"], label="All new brands")

fig, ax = plt.subplots(figsize=(7, 4.2))
kmf.plot_survival_function(ax=ax, color=PALETTE[0])
ax.set_xlabel("Months since first leak-site post"); ax.set_ylabel("Share of brands still active")
ax.set_title("Survival of ransomware brands (Kaplan–Meier)")
savefig(fig, "fig5_brand_survival"); plt.show()

print(f"Median survival: {kmf.median_survival_time_:.1f} months (n={len(surv)}, events={int(surv['exited'].sum())})")
life.sort_values("victims", ascending=False).head(15)

## 6. Takedown effects: LockBit (Operation Cronos)

Two questions: (a) does the total number of claims fall after the takedown? (b) does LockBit's share move to other brands (**redistribution**, consistent with migrating affiliates)?

For (a) we fit an **interrupted time series** on monthly counts, `victims ~ t + post + t_post`, with HAC (Newey–West) standard errors. With few points and a strong trend it is exploratory, so we also run a placebo (fake date) and the pooled seven-operation event study in `robustness_step2.py`.

After Cronos, LockBit reposted old victims; the `group + victim_key` deduplication in section 2 prevents counting them twice.

In [ ]:
TARGET = "lockbit"
EVENT = EVENTS["LockBit / Op. Cronos"]
WINDOW = 12   # months before/after

m = df.groupby(["month", "group"]).size().rename("n").reset_index()
tot = m.groupby("month")["n"].sum()
tgt = m[m["group"] == TARGET].set_index("month")["n"].reindex(tot.index, fill_value=0)
es = pd.DataFrame({"total": tot, "target": tgt})
es["target_share"] = es["target"] / es["total"]
es["rel_month"] = [(p - EVENT.to_period("M")).n for p in es.index]
es = es[es["rel_month"].between(-WINDOW, WINDOW)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(es["rel_month"], es["total"], color="#cfd8e6", label="All groups")
axes[0].bar(es["rel_month"], es["target"], color=PALETTE[1], label=TARGET)
axes[0].set_title("Monthly victims around the takedown"); axes[0].legend()
axes[1].plot(es["rel_month"], es["target_share"] * 100, marker="o", color=PALETTE[1])
axes[1].set_title(f"{TARGET} share of victims (%)")
for ax in axes:
    ax.axvline(0, color="#b5453d", ls="--", lw=1)
    ax.set_xlabel("Months relative to takedown")
savefig(fig, "fig6_takedown_event"); plt.show()

In [ ]:
# (a) Interrupted time series on monthly totals
its = es.reset_index(drop=True).copy()
its["t"] = its["rel_month"]
its["post"] = (its["t"] >= 0).astype(int)
its["t_post"] = its["t"] * its["post"]
its["log_total"] = np.log(its["total"])

for y in ("total", "log_total"):
    res = smf.ols(f"{y} ~ t + post + t_post", data=its).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    print(f"\n=== {y} ===")
    print(res.summary().tables[1])

# Placebo: same specification with a fake event 6 months earlier.
# A significant placebo means the ITS is unreliable: the series has its own shocks (e.g. mass campaigns such as MOVEit/Clop).
plc = its[its["t"] < 0].copy()
plc["t"] = plc["t"] + 6
plc["post"] = (plc["t"] >= 0).astype(int); plc["t_post"] = plc["t"] * plc["post"]
res_p = smf.ols("total ~ t + post + t_post", data=plc).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
print(f"\nPlacebo (−6 months): post coef. = {res_p.params['post']:.1f}, p = {res_p.pvalues['post']:.3f}")

In [ ]:
# (b) Redistribution: who gains share in the 6 months after vs the 6 months before?
H = 6
ev_m = EVENT.to_period("M")
fc = df[df["first_claim"]]          # excludes reposts of others' leaks (e.g. Dispossessor)
pre = fc[(fc["month"] >= ev_m - H) & (fc["month"] < ev_m)]
post = fc[(fc["month"] > ev_m) & (fc["month"] <= ev_m + H)]
shares = pd.DataFrame({
    "share_pre": pre["group"].value_counts(normalize=True),
    "share_post": post["group"].value_counts(normalize=True),
}).fillna(0) * 100
shares["delta_pp"] = shares["share_post"] - shares["share_pre"]
shares["new_brand"] = ~shares.index.isin(df.loc[df["month"] < ev_m, "group"].unique())
redistribution = shares.sort_values("delta_pp")
show = pd.concat([redistribution.head(5), redistribution.tail(10)])

fig, ax = plt.subplots(figsize=(7, 5))
colors = [PALETTE[7] if d < 0 else PALETTE[2] for d in show["delta_pp"]]
ax.barh(show.index, show["delta_pp"], color=colors)
ax.axvline(0, color="#333", lw=0.8)
ax.set_xlabel("Change in share of first-time claims (pp), 6 months after vs before")
ax.set_title("Redistribution of market share after the LockBit takedown")
savefig(fig, "fig7_redistribution"); plt.show()
show.round(2)

## 7. Sectors and countries

Focus on the EU-27 (relevant for NIS2) and Italy. A group's **specialisation** is the Jensen–Shannon divergence between its sector mix and the market's (0 = same mix as the market, 1 = completely different), computed only for groups with enough claims of known sector.

In [ ]:
EU27 = {"AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","GR","HU","IE","IT",
        "LV","LT","LU","MT","NL","PL","PT","RO","SK","SI","ES","SE"}
df["eu"] = df["country"].isin(EU27)

geo = df.dropna(subset=["country"]).groupby(df["discovered"].dt.year).agg(
    victims=("victim_key", "size"),
    eu_share=("eu", "mean"),
    italy=("country", lambda s: (s == "IT").sum()),
    us_share=("country", lambda s: (s == "US").mean()),
)
display(geo.style.format({"eu_share": "{:.1%}", "us_share": "{:.1%}"}))

top_countries = df["country"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(7, 4.5))
cols = [PALETTE[1] if c == "IT" else (PALETTE[0] if c in EU27 else "#9aa5b1") for c in top_countries.index]
ax.barh(top_countries.index[::-1], top_countries.values[::-1], color=cols[::-1])
ax.set_xlabel("Victims (whole period)"); ax.set_title("Top 15 countries (blue = EU, orange = Italy)")
savefig(fig, "fig8_countries"); plt.show()

In [ ]:
from scipy.spatial.distance import jensenshannon

MIN_SECTOR_OBS = 50
known = df.dropna(subset=["sector"])
print(f"Victims with known sector: {len(known) / len(df):.1%}")

top_sectors = known["sector"].value_counts().head(12).index
known = known.assign(sector_g=np.where(known["sector"].isin(top_sectors), known["sector"], "Other"))
market_mix = known["sector_g"].value_counts(normalize=True)

ct = pd.crosstab(known["group"], known["sector_g"])
ct = ct[ct.sum(axis=1) >= MIN_SECTOR_OBS]
mix = ct.div(ct.sum(axis=1), axis=0)[market_mix.index]
spec = pd.DataFrame({
    "victims_with_sector": ct.sum(axis=1),
    "js_divergence": [jensenshannon(mix.loc[g], market_mix, base=2) for g in mix.index],
    "top_sector": mix.idxmax(axis=1),
    "top_sector_share": mix.max(axis=1),
}).sort_values("js_divergence", ascending=False)

# Heatmap: ratio of the sector's share in the group to its share in the market (>1 = over-represented)
lift = (mix / market_mix).loc[spec.index[:15]]
fig, ax = plt.subplots(figsize=(10, 5.5))
im = ax.imshow(np.log2(lift.clip(lower=1/8, upper=8)), cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
ax.set_xticks(range(len(lift.columns))); ax.set_xticklabels(lift.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(lift.index))); ax.set_yticklabels(lift.index, fontsize=8)
cb = fig.colorbar(im, ax=ax); cb.set_label("log2(group share / market share)")
ax.set_title("Sector over/under-representation by group (most specialised first)")
ax.grid(False)
savefig(fig, "fig9_sector_specialisation"); plt.show()
spec.round(3).head(15)

## 8. Export

In [ ]:
df.drop(columns=["month", "quarter"]).to_csv(DATA_DIR / "victims_clean.csv.gz", index=False)
conc.to_csv(TAB_DIR / "concentration_quarterly.csv")
life.to_csv(TAB_DIR / "brand_lifetimes.csv")
redistribution.to_csv(TAB_DIR / "takedown_redistribution.csv")
spec.to_csv(TAB_DIR / "sector_specialisation.csv")
geo.to_csv(TAB_DIR / "geography_yearly.csv")
print("Saved:", sorted(p.name for p in FIG_DIR.glob("*.png")), sorted(p.name for p in TAB_DIR.glob("*.csv")))

## Possible extensions

- Weight claims by victim size (revenue or employees) where available, to get a unit closer to economic impact.

## 9. Robustness

After the export, `paper_stats.py` produces the statistics cited in the text and `robustness_step2.py` the robustness analyses: CR_k for the top 1/5/10/20% of groups (comparison with Müller and Yannikos 2026), HHI and survival with documented rebrands merged into one organisation, survival with exit thresholds of 60/90/180 days, and an event study of seven law-enforcement operations with randomisation inference. `compare_ransomwatch.py` compares coverage with the ransomwatch dataset (github.com/joshhighet/ransomwatch, `posts.json`).

In [ ]:
%run paper_stats.py
%run robustness_step2.py
# coverage comparison: first download ransomwatch posts.json into data/ext/
# %run compare_ransomwatch.py data/ext/rw_posts.json data/victims_clean.csv.gz tables/coverage_ransomwatch_quarterly.csv 2025-01-01